# 🧠 Neural Network Regression Tutorial

## Learning Objectives

By the end of this tutorial, you will:
- Understand what Neural Networks are and how they work
- Learn about layers, neurons, and activation functions
- Build neural networks with 1 layer, 2 layers, and deeper architectures
- Understand the difference between shallow and deep networks
- Train neural networks for regression problems
- Compare performance of different network architectures
- Visualize training progress and predictions

---

## What is a Neural Network?

A **Neural Network** is a machine learning model inspired by how the human brain works.

### The Basic Idea:

Neural networks consist of:
- **Input Layer**: Receives your data (features)
- **Hidden Layers**: Process the data through multiple transformations
- **Output Layer**: Produces predictions

### Key Components:

1. **Neurons (Nodes)**: Basic processing units
2. **Weights**: Parameters that the model learns
3. **Biases**: Additional parameters for flexibility
4. **Activation Functions**: Non-linear functions that enable complex patterns

### Why Use Neural Networks?

- **Non-linear Relationships**: Can learn complex patterns that linear models cannot
- **Flexibility**: Can model very complex functions
- **Scalability**: Work well with large datasets
- **Universal Approximation**: Can approximate any continuous function (with enough neurons)

---

## Understanding Network Depth

### 1-Layer Network (Shallow):
- Input → Hidden Layer → Output
- Good for simple patterns
- Fast to train

### 2-Layer Network:
- Input → Hidden Layer 1 → Hidden Layer 2 → Output
- Can learn more complex patterns
- Better for non-linear relationships

### 3+ Layer Network (Deep):
- Multiple hidden layers
- Can learn very complex patterns
- Requires more data and computation

---

## Step 1: Import Required Libraries

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle, FancyBboxPatch, FancyArrowPatch
from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras for neural networks
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set visualization style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')

sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## Step 1.5: Visualize Neural Network Architecture

Let's create a function to visualize how neural networks are structured!

In [ ]:
def draw_neural_network(ax, layer_sizes, title="Neural Network Architecture"):
    """
    Draw a neural network architecture diagram
    
    Parameters:
    -----------
    ax : matplotlib axis
        Axis to draw on
    layer_sizes : list
        List of integers representing neurons in each layer
        Example: [5, 64, 32, 1] means 5 inputs, 64 hidden, 32 hidden, 1 output
    title : str
        Title for the diagram
    """
    ax.set_xlim(0, len(layer_sizes))
    ax.set_ylim(0, max(layer_sizes) + 2)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Colors for different layer types
    colors = {
        'input': '#4A90E2',      # Blue
        'hidden': '#50C878',      # Green
        'output': '#FF6B6B'       # Red
    }
    
    # Draw layers
    for layer_idx, num_neurons in enumerate(layer_sizes):
        x_pos = layer_idx + 0.5
        layer_type = 'input' if layer_idx == 0 else ('output' if layer_idx == len(layer_sizes) - 1 else 'hidden')
        color = colors[layer_type]
        
        # Draw neurons in this layer
        y_spacing = max(layer_sizes) / (num_neurons + 1)
        for neuron_idx in range(num_neurons):
            y_pos = (neuron_idx + 1) * y_spacing + 0.5
            
            # Draw neuron circle
            circle = Circle((x_pos, y_pos), 0.15, color=color, ec='black', lw=1.5, zorder=3)
            ax.add_patch(circle)
            
            # Draw connections to next layer
            if layer_idx < len(layer_sizes) - 1:
                next_num_neurons = layer_sizes[layer_idx + 1]
                next_y_spacing = max(layer_sizes) / (next_num_neurons + 1)
                for next_neuron_idx in range(next_num_neurons):
                    next_y_pos = (next_neuron_idx + 1) * next_y_spacing + 0.5
                    # Draw connection line
                    ax.plot([x_pos + 0.15, x_pos + 0.85], [y_pos, next_y_pos], 
                           'gray', alpha=0.3, linewidth=0.5, zorder=1)
        
        # Add layer label
        layer_name = 'Input' if layer_idx == 0 else ('Output' if layer_idx == len(layer_sizes) - 1 else f'Hidden {layer_idx}')
        ax.text(x_pos, max(layer_sizes) + 1.2, f'{layer_name}\\n({num_neurons} neurons)', 
               ha='center', va='bottom', fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='black', alpha=0.8))

# Create visualization for different architectures
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Neural Network Architecture Comparison', fontsize=16, fontweight='bold')

# 1-Layer Network (5 inputs -> 64 hidden -> 1 output)
draw_neural_network(axes[0, 0], [5, 64, 1], '1-Layer Network\\n(Input → Hidden → Output)')

# 2-Layer Network (5 inputs -> 64 hidden -> 32 hidden -> 1 output)
draw_neural_network(axes[0, 1], [5, 64, 32, 1], '2-Layer Network\\n(Input → Hidden1 → Hidden2 → Output)')

# 3-Layer Network (5 inputs -> 128 hidden -> 64 hidden -> 32 hidden -> 1 output)
draw_neural_network(axes[1, 0], [5, 128, 64, 32, 1], '3-Layer Network (Deep)\\n(Input → Hidden1 → Hidden2 → Hidden3 → Output)')

# 4-Layer Network (5 inputs -> 128 hidden -> 64 hidden -> 32 hidden -> 16 hidden -> 1 output)
draw_neural_network(axes[1, 1], [5, 128, 64, 32, 16, 1], '4-Layer Network (Deeper)\\n(Input → Hidden1 → Hidden2 → Hidden3 → Hidden4 → Output)')

plt.tight_layout()
plt.show()

print("✅ Neural network architecture visualizations created!")
print("\\n💡 Notice how:")
print("   - More layers = More complex transformations")
print("   - Each connection represents a weight that the model learns")
print("   - Neurons process information and pass it to the next layer")

## Step 2: Create Sample Regression Dataset

We'll create a dataset with non-linear relationships to demonstrate the power of neural networks.

## Step 2: Create Sample Regression Dataset

We'll create a dataset with non-linear relationships to demonstrate the power of neural networks.

## Step 3.5: Visualize Model Complexity

Let's compare the complexity of different network architectures by looking at the number of parameters (weights and biases) each model has.

In [ ]:
# Calculate and visualize model complexity
def count_parameters(model):
    """Count total number of trainable parameters in a model"""
    return model.count_params()

# We'll calculate this after models are built, but let's create a placeholder visualization
# showing the concept of model complexity

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Understanding Neural Network Complexity', fontsize=16, fontweight='bold')

# Example parameter counts (approximate for our architectures)
architectures = ['1-Layer\\n(64 neurons)', '2-Layer\\n(64→32)', '3-Layer\\n(128→64→32)', '4-Layer\\n(128→64→32→16)']
approx_params = [385, 2241, 12417, 13825]  # Approximate parameter counts

# Bar chart of parameters
axes[0].bar(architectures, approx_params, color=['blue', 'green', 'orange', 'purple'], 
           alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Number of Parameters', fontsize=12, fontweight='bold')
axes[0].set_title('Model Complexity: Number of Trainable Parameters', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(approx_params):
    axes[0].text(i, v + max(approx_params)*0.02, f'{v:,}', ha='center', fontweight='bold', fontsize=10)

# Training time comparison (conceptual)
training_time = [1.0, 1.3, 2.1, 2.8]  # Relative training time
axes[1].bar(architectures, training_time, color=['blue', 'green', 'orange', 'purple'], 
           alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Relative Training Time', fontsize=12, fontweight='bold')
axes[1].set_title('Training Time Comparison (Relative)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(training_time):
    axes[1].text(i, v + max(training_time)*0.05, f'{v:.1f}x', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print("✅ Model complexity visualization created!")
print("\\n💡 Key Insights:")
print("   - More layers = More parameters to learn")
print("   - More parameters = More complex patterns can be learned")
print("   - But also: More data needed, longer training time")
print("   - Balance complexity with your data size and needs!")

In [ ]:
# Create a non-linear regression dataset
np.random.seed(42)
n_samples = 1000

# Generate features
X = np.random.randn(n_samples, 5)  # 5 features

# Create non-linear target with multiple features
# y = 2*x1^2 + 3*x2*x3 + sin(x4) + 0.5*x5 + noise
y = (2 * X[:, 0]**2 + 
     3 * X[:, 1] * X[:, 2] + 
     np.sin(X[:, 3]) + 
     0.5 * X[:, 4] + 
     np.random.normal(0, 0.1, n_samples))

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the features (important for neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Dataset created!")
print(f"Training samples: {X_train_scaled.shape[0]}")
print(f"Test samples: {X_test_scaled.shape[0]}")
print(f"Features: {X_train_scaled.shape[1]}")
print(f"Target range: [{y.min():.2f}, {y.max():.2f}]")

## Step 3: Build and Train 1-Layer Neural Network

A 1-layer network has:
- Input layer (your features)
- One hidden layer with neurons
- Output layer (prediction)

In [ ]:
# Build 1-layer neural network
model_1layer = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],), name='hidden_layer_1'),
    layers.Dense(1, name='output_layer')  # Output layer for regression
])

# Compile the model
model_1layer.compile(
    optimizer='adam',
    loss='mse',  # Mean Squared Error for regression
    metrics=['mae']  # Mean Absolute Error as additional metric
)

# Display model architecture
print("📊 1-Layer Neural Network Architecture:")
print("=" * 60)
model_1layer.summary()

In [ ]:
# Train the 1-layer model
print("🚀 Training 1-Layer Neural Network...")
print("=" * 60)

history_1layer = model_1layer.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
)

print("\n✅ Training completed!")

In [ ]:
# Evaluate 1-layer model
y_pred_1layer = model_1layer.predict(X_test_scaled, verbose=0)

mse_1layer = mean_squared_error(y_test, y_pred_1layer)
mae_1layer = mean_absolute_error(y_test, y_pred_1layer)
rmse_1layer = np.sqrt(mse_1layer)
r2_1layer = r2_score(y_test, y_pred_1layer)

print("📊 1-Layer Network Performance:")
print("=" * 60)
print(f"MSE:  {mse_1layer:.4f}")
print(f"MAE:  {mae_1layer:.4f}")
print(f"RMSE: {rmse_1layer:.4f}")
print(f"R²:   {r2_1layer:.4f}")

## Step 4: Build and Train 2-Layer Neural Network

A 2-layer network has:
- Input layer
- First hidden layer
- Second hidden layer
- Output layer

In [ ]:
# Build 2-layer neural network
model_2layer = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],), name='hidden_layer_1'),
    layers.Dense(32, activation='relu', name='hidden_layer_2'),
    layers.Dense(1, name='output_layer')
])

# Compile the model
model_2layer.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Display model architecture
print("📊 2-Layer Neural Network Architecture:")
print("=" * 60)
model_2layer.summary()

In [ ]:
# Train the 2-layer model
print("🚀 Training 2-Layer Neural Network...")
print("=" * 60)

history_2layer = model_2layer.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
)

print("\n✅ Training completed!")

In [ ]:
# Evaluate 2-layer model
y_pred_2layer = model_2layer.predict(X_test_scaled, verbose=0)

mse_2layer = mean_squared_error(y_test, y_pred_2layer)
mae_2layer = mean_absolute_error(y_test, y_pred_2layer)
rmse_2layer = np.sqrt(mse_2layer)
r2_2layer = r2_score(y_test, y_pred_2layer)

print("📊 2-Layer Network Performance:")
print("=" * 60)
print(f"MSE:  {mse_2layer:.4f}")
print(f"MAE:  {mae_2layer:.4f}")
print(f"RMSE: {rmse_2layer:.4f}")
print(f"R²:   {r2_2layer:.4f}")

## Step 5: Build and Train 3-Layer Neural Network (Deep Network)

A 3-layer network has:
- Input layer
- Three hidden layers
- Output layer

In [ ]:
# Build 3-layer neural network (deep network)
model_3layer = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],), name='hidden_layer_1'),
    layers.Dense(64, activation='relu', name='hidden_layer_2'),
    layers.Dense(32, activation='relu', name='hidden_layer_3'),
    layers.Dense(1, name='output_layer')
])

# Compile the model
model_3layer.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Display model architecture
print("📊 3-Layer Neural Network Architecture (Deep Network):")
print("=" * 60)
model_3layer.summary()

In [ ]:
# Train the 3-layer model
print("🚀 Training 3-Layer Neural Network (Deep Network)...")
print("=" * 60)

history_3layer = model_3layer.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
)

print("\n✅ Training completed!")

In [ ]:
# Evaluate 3-layer model
y_pred_3layer = model_3layer.predict(X_test_scaled, verbose=0)

mse_3layer = mean_squared_error(y_test, y_pred_3layer)
mae_3layer = mean_absolute_error(y_test, y_pred_3layer)
rmse_3layer = np.sqrt(mse_3layer)
r2_3layer = r2_score(y_test, y_pred_3layer)

print("📊 3-Layer Network Performance:")
print("=" * 60)
print(f"MSE:  {mse_3layer:.4f}")
print(f"MAE:  {mae_3layer:.4f}")
print(f"RMSE: {rmse_3layer:.4f}")
print(f"R²:   {r2_3layer:.4f}")

## Step 6: Build 4-Layer Network (Even Deeper)

Let's try an even deeper network to see how depth affects performance.

In [ ]:
# Note: Parameter counting will be done after all models are built
# See the cell after Step 6 (after model_4layer is created) for actual parameter counts
print("💡 Parameter counting visualization will be shown after all models are built.")
print("   This ensures all models (including 4-layer) are available for comparison.")

In [ ]:
# Build 4-layer neural network
model_4layer = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],), name='hidden_layer_1'),
    layers.Dense(64, activation='relu', name='hidden_layer_2'),
    layers.Dense(32, activation='relu', name='hidden_layer_3'),
    layers.Dense(16, activation='relu', name='hidden_layer_4'),
    layers.Dense(1, name='output_layer')
])

# Compile the model
model_4layer.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Display model architecture
print("📊 4-Layer Neural Network Architecture:")
print("=" * 60)
model_4layer.summary()

In [ ]:
# Train the 4-layer model
print("🚀 Training 4-Layer Neural Network...")
print("=" * 60)

history_4layer = model_4layer.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
)

print("\n✅ Training completed!")

In [ ]:
# Evaluate 4-layer model
y_pred_4layer = model_4layer.predict(X_test_scaled, verbose=0)

mse_4layer = mean_squared_error(y_test, y_pred_4layer)
mae_4layer = mean_absolute_error(y_test, y_pred_4layer)
rmse_4layer = np.sqrt(mse_4layer)
r2_4layer = r2_score(y_test, y_pred_4layer)

print("📊 4-Layer Network Performance:")
print("=" * 60)
print(f"MSE:  {mse_4layer:.4f}")
print(f"MAE:  {mae_4layer:.4f}")
print(f"RMSE: {rmse_4layer:.4f}")
print(f"R²:   {r2_4layer:.4f}")

## Step 6.5: Visualize Actual Model Complexity

Now that all models are built, let's see the actual number of parameters in each model.

In [ ]:
# After all models are built, let's visualize actual parameter counts
print("📊 Actual Model Complexity (Parameter Counts):")
print("=" * 60)

models_info = [
    ('1-Layer', model_1layer),
    ('2-Layer', model_2layer),
    ('3-Layer', model_3layer),
    ('4-Layer', model_4layer)
]

param_counts = []
model_names = []

for name, model in models_info:
    params = count_parameters(model)
    param_counts.append(params)
    model_names.append(name)
    print(f"{name:12} : {params:>8,} parameters")

# Visualize actual parameter counts
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
bars = ax.bar(model_names, param_counts, color=['blue', 'green', 'orange', 'purple'], 
             alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Number of Parameters', fontsize=12, fontweight='bold')
ax.set_title('Actual Model Complexity: Trainable Parameters', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, count) in enumerate(zip(bars, param_counts)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + max(param_counts)*0.01,
            f'{count:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print("\n✅ Parameter count visualization created!")

## Step 7: Compare All Models

In [ ]:
# Create comparison table
comparison_data = {
    'Model': ['1-Layer', '2-Layer', '3-Layer', '4-Layer'],
    'MSE': [mse_1layer, mse_2layer, mse_3layer, mse_4layer],
    'MAE': [mae_1layer, mae_2layer, mae_3layer, mae_4layer],
    'RMSE': [rmse_1layer, rmse_2layer, rmse_3layer, rmse_4layer],
    'R²': [r2_1layer, r2_2layer, r2_3layer, r2_4layer]
}

comparison_df = pd.DataFrame(comparison_data)

print("📊 Model Comparison:")
print("=" * 80)
print(comparison_df.to_string(index=False))

# Find best model
best_model_idx = comparison_df['R²'].idxmax()
best_model = comparison_df.loc[best_model_idx, 'Model']
print(f"\n🏆 Best Model: {best_model} (R² = {comparison_df.loc[best_model_idx, 'R²']:.4f})")

## Step 8: Visualize Training Progress

In [ ]:
# Plot training history for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Training Progress: Loss and MAE Over Epochs', fontsize=16, fontweight='bold')

# Loss comparison
axes[0, 0].plot(history_1layer.history['loss'], label='1-Layer (Train)', linestyle='-', linewidth=2)
axes[0, 0].plot(history_1layer.history['val_loss'], label='1-Layer (Val)', linestyle='--', linewidth=2)
axes[0, 0].plot(history_2layer.history['loss'], label='2-Layer (Train)', linestyle='-', linewidth=2)
axes[0, 0].plot(history_2layer.history['val_loss'], label='2-Layer (Val)', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Loss: 1-Layer vs 2-Layer', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history_3layer.history['loss'], label='3-Layer (Train)', linestyle='-', linewidth=2)
axes[0, 1].plot(history_3layer.history['val_loss'], label='3-Layer (Val)', linestyle='--', linewidth=2)
axes[0, 1].plot(history_4layer.history['loss'], label='4-Layer (Train)', linestyle='-', linewidth=2)
axes[0, 1].plot(history_4layer.history['val_loss'], label='4-Layer (Val)', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Loss: 3-Layer vs 4-Layer', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# MAE comparison
axes[1, 0].plot(history_1layer.history['mae'], label='1-Layer (Train)', linestyle='-', linewidth=2)
axes[1, 0].plot(history_1layer.history['val_mae'], label='1-Layer (Val)', linestyle='--', linewidth=2)
axes[1, 0].plot(history_2layer.history['mae'], label='2-Layer (Train)', linestyle='-', linewidth=2)
axes[1, 0].plot(history_2layer.history['val_mae'], label='2-Layer (Val)', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('MAE', fontsize=12, fontweight='bold')
axes[1, 0].set_title('MAE: 1-Layer vs 2-Layer', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history_3layer.history['mae'], label='3-Layer (Train)', linestyle='-', linewidth=2)
axes[1, 1].plot(history_3layer.history['val_mae'], label='3-Layer (Val)', linestyle='--', linewidth=2)
axes[1, 1].plot(history_4layer.history['mae'], label='4-Layer (Train)', linestyle='-', linewidth=2)
axes[1, 1].plot(history_4layer.history['val_mae'], label='4-Layer (Val)', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('MAE', fontsize=12, fontweight='bold')
axes[1, 1].set_title('MAE: 3-Layer vs 4-Layer', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Training progress visualization created!")

## Step 9.5: Visualize Activation Functions

Understanding activation functions helps us understand how neural networks learn non-linear patterns.

In [ ]:
# Visualize activation functions
x = np.linspace(-5, 5, 100)

# ReLU (Rectified Linear Unit) - most common
relu = np.maximum(0, x)

# Sigmoid
sigmoid = 1 / (1 + np.exp(-x))

# Tanh
tanh = np.tanh(x)

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Common Activation Functions in Neural Networks', fontsize=16, fontweight='bold')

# ReLU
axes[0].plot(x, relu, linewidth=3, color='blue', label='ReLU')
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[0].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Input (x)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Output f(x)', fontsize=12, fontweight='bold')
axes[0].set_title('ReLU: f(x) = max(0, x)\\n(Used in our networks)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

# Sigmoid
axes[1].plot(x, sigmoid, linewidth=3, color='green', label='Sigmoid')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='y=0.5')
axes[1].set_xlabel('Input (x)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Output f(x)', fontsize=12, fontweight='bold')
axes[1].set_title('Sigmoid: f(x) = 1/(1+e⁻ˣ)\\n(Output between 0 and 1)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)
axes[1].set_ylim([-0.1, 1.1])

# Tanh
axes[2].plot(x, tanh, linewidth=3, color='orange', label='Tanh')
axes[2].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[2].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[2].set_xlabel('Input (x)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Output f(x)', fontsize=12, fontweight='bold')
axes[2].set_title('Tanh: f(x) = tanh(x)\\n(Output between -1 and 1)', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].legend(fontsize=11)
axes[2].set_ylim([-1.1, 1.1])

plt.tight_layout()
plt.show()

print("✅ Activation function visualizations created!")
print("\\n💡 Key Points:")
print("   - ReLU: Simple, fast, prevents vanishing gradients")
print("   - Sigmoid: Squashes values to 0-1 range")
print("   - Tanh: Squashes values to -1 to 1 range")
print("   - Without activation functions, neural networks would only learn linear relationships!")

In [ ]:
# Plot predictions vs actual values
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Predictions vs Actual Values', fontsize=16, fontweight='bold')

# 1-Layer
axes[0, 0].scatter(y_test, y_pred_1layer, alpha=0.6, s=30, color='blue')
min_val = min(y_test.min(), y_pred_1layer.min())
max_val = max(y_test.max(), y_pred_1layer.max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Values', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Predicted Values', fontsize=12, fontweight='bold')
axes[0, 0].set_title(f'1-Layer Network (R² = {r2_1layer:.4f})', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2-Layer
axes[0, 1].scatter(y_test, y_pred_2layer, alpha=0.6, s=30, color='green')
min_val = min(y_test.min(), y_pred_2layer.min())
max_val = max(y_test.max(), y_pred_2layer.max())
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 1].set_xlabel('Actual Values', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Predicted Values', fontsize=12, fontweight='bold')
axes[0, 1].set_title(f'2-Layer Network (R² = {r2_2layer:.4f})', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3-Layer
axes[1, 0].scatter(y_test, y_pred_3layer, alpha=0.6, s=30, color='orange')
min_val = min(y_test.min(), y_pred_3layer.min())
max_val = max(y_test.max(), y_pred_3layer.max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1, 0].set_xlabel('Actual Values', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Predicted Values', fontsize=12, fontweight='bold')
axes[1, 0].set_title(f'3-Layer Network (R² = {r2_3layer:.4f})', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4-Layer
axes[1, 1].scatter(y_test, y_pred_4layer, alpha=0.6, s=30, color='purple')
min_val = min(y_test.min(), y_pred_4layer.min())
max_val = max(y_test.max(), y_pred_4layer.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1, 1].set_xlabel('Actual Values', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Predicted Values', fontsize=12, fontweight='bold')
axes[1, 1].set_title(f'4-Layer Network (R² = {r2_4layer:.4f})', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Predictions visualization created!")

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

# R² comparison
models = ['1-Layer', '2-Layer', '3-Layer', '4-Layer']
r2_scores = [r2_1layer, r2_2layer, r2_3layer, r2_4layer]
colors = ['blue', 'green', 'orange', 'purple']

axes[0].bar(models, r2_scores, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('R² Score', fontsize=12, fontweight='bold')
axes[0].set_title('R² Score Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylim([min(r2_scores) - 0.05, max(r2_scores) + 0.05])
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(r2_scores):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# RMSE comparison
rmse_scores = [rmse_1layer, rmse_2layer, rmse_3layer, rmse_4layer]

axes[1].bar(models, rmse_scores, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('RMSE', fontsize=12, fontweight='bold')
axes[1].set_title('RMSE Comparison (Lower is Better)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(rmse_scores):
    axes[1].text(i, v + max(rmse_scores)*0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Model comparison visualization created!")

## 🎓 Summary and Key Takeaways

### ✅ What You've Learned:

1. **Neural Network Basics**:
   - Input layer, hidden layers, output layer
   - Neurons, weights, biases, activation functions

2. **Network Depth**:
   - **1-Layer**: Simple, fast, good for basic patterns
   - **2-Layer**: Better for non-linear relationships
   - **3+ Layers**: Deep networks for complex patterns

3. **Key Concepts**:
   - **Activation Functions**: ReLU enables non-linear learning
   - **Loss Function**: MSE for regression problems
   - **Optimizer**: Adam for efficient training
   - **Early Stopping**: Prevents overfitting

4. **Model Evaluation**:
   - R²: How well the model fits
   - RMSE: Average prediction error
   - MAE: Mean absolute error

### 💡 Important Insights:

- **More layers ≠ Always better**: Sometimes simpler models work just as well
- **Data scaling is crucial**: Neural networks work best with scaled features
- **Early stopping helps**: Prevents overfitting and saves training time
- **Visualization is key**: Always plot predictions vs actual values

### 📚 When to Use Each Architecture:

| Architecture | Best For | Pros | Cons |
|-------------|----------|------|------|
| **1-Layer** | Simple patterns, small datasets | Fast, easy to train | Limited complexity |
| **2-Layer** | Moderate complexity | Good balance | May need tuning |
| **3+ Layers** | Complex patterns, large datasets | Very flexible | Needs more data, slower |

### 🔧 Tips for Building Neural Networks:

1. **Start simple**: Begin with 1-2 layers
2. **Scale your data**: Use StandardScaler or MinMaxScaler
3. **Use early stopping**: Monitor validation loss
4. **Experiment**: Try different architectures
5. **Visualize**: Always check predictions visually

---

**Great job learning neural networks for regression! 🧠✨**